# 3D differentiation: shifted Taylor–Green vortex

This is a spatial differentiation test of a prescribed velocity field, not a
Navier–Stokes time simulation. On the box $[0,3\pi/2]^3$, define
$X=x-0.3$, $Y=y-0.5$, $Z=z-0.7$ and
\[
\mathbf u=(\sin X\cos Y\cos Z,\;-\cos X\sin Y\cos Z,\;0).
\]
A translation alone does **not** remove periodicity on a full $2\pi$ box.
Here the box spans three quarters of a period, so opposite faces do not match;
the underlying trigonometric field is periodic, but its restriction to this box
is nonperiodic boundary data. We explicitly check all three face mismatches.

The analytic identities are $\nabla\cdot\mathbf u=0$,
$\nabla^2\mathbf u=-3\mathbf u$, and
\[
\nabla\times\mathbf u=(-\cos X\sin Y\sin Z,
-\sin X\cos Y\sin Z,2\sin X\sin Y\cos Z).
\]
The full velocity Jacobian is checked too, so divergence cancellation cannot
hide errors in individual derivatives. A rectangular initial grid checks axis
ordering; subsequent cubic grids measure convergence.

Install `python -m pip install -e './jax[notebook]'` from the repository root
and select that environment as the notebook kernel. All differentiation uses
public JAX BSPF operators, with local Chebyshev endpoint estimation.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import bspf_jax as bspf
import matplotlib.pyplot as plt

length = 1.5*jnp.pi
options = dict(degree=9, n_basis=18, lam=1e-6, endpoint_method="chebyshev",
               boundary_points=16, chebyshev_modes=12, chebyshev_alpha=1e-12)

def vortex(x, y, z):
    X, Y, Z = jnp.meshgrid(x-0.3, y-0.5, z-0.7, indexing="ij")
    sx, sy, sz = jnp.sin(X), jnp.sin(Y), jnp.sin(Z)
    cx, cy, cz = jnp.cos(X), jnp.cos(Y), jnp.cos(Z)
    zero = jnp.zeros_like(X)
    velocity = jnp.stack((sx*cy*cz, -cx*sy*cz, zero))
    jacobian = jnp.stack((jnp.stack((cx*cy*cz, sx*sy*cz, zero), axis=-1),
                          jnp.stack((-sx*sy*cz, -cx*cy*cz, zero), axis=-1),
                          jnp.stack((-sx*cy*sz, cx*sy*sz, zero), axis=-1)))
    vorticity = jnp.stack((-cx*sy*sz, -sx*cy*sz, 2*sx*sy*cz))
    return velocity, jacobian, vorticity

relative_l2 = lambda actual, exact: jnp.linalg.norm(actual-exact)/jnp.linalg.norm(exact)
gradient = jax.jit(bspf.gradient)


## Full 3D derivatives and nonperiodic faces

Velocity has shape `(3, nx, ny, nz)` for `curl` and `divergence`.
For batched `gradient` and `laplacian`, move components to the trailing axis.
The Jacobian layout is `(derivative_axis, nx, ny, nz, velocity_component)`.
Divergence has a zero reference, so report its **absolute** maximum error.

**CPU runtime setup:** launch Jupyter with `OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1
python -m jupyterlab`, then start a fresh kernel. The development OpenMP OpenBLAS
build returned incorrect results for concurrent multithreaded solves; one BLAS
thread fixes the observed failure while preserving outer JIT. Setting variables
after importing numerical libraries is not reliable. See
[the diagnosis](../../docs/3d_jit_diagnosis.md).


In [ ]:
x, y, z = (jnp.linspace(0, length, n) for n in (33, 41, 49))
plan = bspf.plan_3d(x, y, z, **options)
velocity, exact_jacobian, exact_curl = vortex(x, y, z)
field = jnp.moveaxis(velocity, 0, -1)
face_gaps = [float(jnp.max(jnp.abs(jnp.take(field, 0, axis=i)-jnp.take(field, -1, axis=i))))
             for i in range(3)]
print("Opposite-face velocity mismatches (x, y, z):", face_gaps)
assert min(face_gaps) > 0.1

jacobian = gradient(plan, field)
div = jax.jit(bspf.divergence)(plan, velocity)
vorticity = jax.jit(bspf.curl)(plan, velocity)
laplacian = jax.jit(bspf.laplacian)(plan, field)
metrics = dict(jacobian_relative_l2=float(relative_l2(jacobian, exact_jacobian)),
               curl_relative_l2=float(relative_l2(vorticity, exact_curl)),
               divergence_max_abs=float(jnp.max(jnp.abs(div))),
               laplacian_relative_l2=float(relative_l2(laplacian, -3*field)))
print(metrics)
assert jacobian.shape == (3, x.size, y.size, z.size, 3)
assert metrics["jacobian_relative_l2"] < 1e-8
assert metrics["curl_relative_l2"] < 1e-8
assert metrics["divergence_max_abs"] < 1e-8
assert metrics["laplacian_relative_l2"] < 1e-7


## Grid refinement

Geometrically spaced sizes are rounded to odd integers. All physical parameters
and estimator settings remain fixed. Only the grid changes. This verifies the
full Jacobian, including derivatives in z despite the zero vertical velocity.


In [ ]:
sizes = sorted({2*round((float(n)-1)/2)+1 for n in jnp.geomspace(17, 65, 5)})
errors = []
for n in sizes:
    coords = jnp.linspace(0, length, n)
    pn = bspf.plan_3d(coords, coords, coords, **options)
    un, reference, _ = vortex(coords, coords, coords)
    numerical = gradient(pn, jnp.moveaxis(un, 0, -1))
    errors.append(float(relative_l2(numerical, reference)))
    print(f"{n}³: Jacobian relative L2 = {errors[-1]:.6e}")
    jax.clear_caches()  # Bound compilation-cache memory across grid shapes.
assert errors[-1] < errors[0]/100


In [ ]:
mid = z.size//2
jacobian_error = jnp.sqrt(jnp.sum(jnp.abs(jacobian-exact_jacobian)**2, axis=(0, 4)))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, values, title in zip(axes[:2], (exact_jacobian[0, :, :, mid, 0], jacobian_error[:, :, mid]),
                              ("Exact du/dx", "Jacobian absolute error (Frobenius norm)")):
    image = ax.pcolormesh(x, y, values.T, shading="auto")
    ax.set(xlabel="x", ylabel="y", title=f"{title}\nz = {float(z[mid]):.3f}")
    fig.colorbar(image, ax=ax)
axes[2].loglog(sizes, errors, "o-")
axes[2].set(xlabel="Points per axis N", ylabel="Jacobian relative L2 error", title="3D convergence")
axes[2].set_xticks(sizes, [str(n) for n in sizes])
axes[2].minorticks_off()
axes[2].grid(True, alpha=0.3)
plt.show()
